# Statistical Baseline: ARIMA for Short-Term Energy Forecasting

In [ ]:
import importlib.util
import os
import random
import subprocess
import sys
import warnings

required_packages = {
    "numpy": "numpy",
    "pandas": "pandas",
    "statsmodels": "statsmodels",
    "sklearn": "scikit-learn",
    "matplotlib": "matplotlib"
}

for module_name, package_name in required_packages.items():
    if importlib.util.find_spec(module_name) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package_name])

warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.stattools import adfuller

np.random.seed(42)
random.seed(42)

In [ ]:
candidate_paths = [
    "/kaggle/input/total-consumption-data/total_consumption_data.csv",
    "/kaggle/input/datasets/rahulsarav/total-consumption-data/total_consumption_data.csv",
    "/kaggle/input/total_consumption_data.csv",
    "total_consumption_data.csv"
]

data_path = None
for p in candidate_paths:
    if os.path.exists(p):
        data_path = p
        break

if data_path is None:
    raise FileNotFoundError("total_consumption_data.csv was not found in expected Kaggle/local paths")

df = pd.read_csv(data_path)
df.columns = [c.strip() for c in df.columns]

if np.issubdtype(df["datetime"].dtype, np.number):
    df["datetime"] = pd.to_datetime(df["datetime"], unit="s", errors="coerce")
else:
    df["datetime"] = pd.to_datetime(df["datetime"], errors="coerce")

for col in df.columns:
    if col not in ["datetime", "meter_id"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

target_col = "apower_ph1" if "apower_ph1" in df.columns else "apower"
df = df.dropna(subset=["datetime", target_col]).copy()
df["hour"] = df["datetime"].dt.floor("h")

series = df.groupby("hour")[target_col].sum().sort_index().astype(float)

n = len(series)
train_end = int(n * 0.70)
val_end = int(n * 0.85)

train = series.iloc[:train_end]
val = series.iloc[train_end:val_end]
test = series.iloc[val_end:]

print("Data path:", data_path)
print("Target column:", target_col)
print("Series length:", len(series))
print("Train/Val/Test lengths:", len(train), len(val), len(test))
series.head()

In [ ]:
def pick_d(series_in, max_d=2, alpha=0.05):
    pvals = {}
    for d in range(max_d + 1):
        s = series_in.diff(d).dropna() if d > 0 else series_in.dropna()
        if len(s) < 30:
            continue
        p = adfuller(s, autolag="AIC")[1]
        pvals[d] = float(p)
        if p < alpha:
            return d, pvals
    if len(pvals) == 0:
        return 0, pvals
    return min(pvals, key=pvals.get), pvals

def metric_row(name, y_true, y_pred):
    yt = np.asarray(y_true).reshape(-1)
    yp = np.asarray(y_pred).reshape(-1)
    return {
        "Model": name,
        "MAE": float(mean_absolute_error(yt, yp)),
        "RMSE": float(np.sqrt(mean_squared_error(yt, yp))),
        "R2": float(r2_score(yt, yp)),
        "MAPE(%)": float(np.mean(np.abs((yt - yp) / (np.abs(yt) + 1e-6))) * 100.0)
    }

d_order, pvals = pick_d(train, max_d=2, alpha=0.05)
print("Chosen differencing order:", d_order)
print("ADF p-values:", pvals)

orders = [(p, d_order, q) for p in range(0, 6) for q in range(0, 6)]
orders = random.sample(orders, k=min(24, len(orders)))

best_order = None
best_val_rmse = np.inf

for order in orders:
    try:
        m = ARIMA(train, order=order, enforce_stationarity=False, enforce_invertibility=False).fit()
        pred_val = m.forecast(steps=len(val))
        val_rmse = float(np.sqrt(mean_squared_error(val.values, pred_val.values)))
        if val_rmse < best_val_rmse:
            best_val_rmse = val_rmse
            best_order = order
    except Exception:
        pass

if best_order is None:
    best_order = (1, d_order, 1)

print("Best ARIMA order on validation:", best_order, "RMSE:", round(best_val_rmse, 4))

In [ ]:
train_val = pd.concat([train, val])
final_model = ARIMA(train_val, order=best_order, enforce_stationarity=False, enforce_invertibility=False).fit()
pred_test = final_model.forecast(steps=len(test))

metrics_df = pd.DataFrame([metric_row("ARIMA", test.values, pred_test.values)])
display(metrics_df.round(4))

pred_frame = pd.DataFrame({
    "datetime": test.index,
    "actual": test.values,
    "pred_arima": pred_test.values
})

plot_n = min(300, len(pred_frame))
plt.figure(figsize=(14, 5))
plt.plot(pred_frame["datetime"].iloc[:plot_n], pred_frame["actual"].iloc[:plot_n], label="Actual", linewidth=1.8)
plt.plot(pred_frame["datetime"].iloc[:plot_n], pred_frame["pred_arima"].iloc[:plot_n], label="ARIMA", linewidth=1.5)
plt.title("ARIMA Forecast vs Actual")
plt.xlabel("Datetime")
plt.ylabel(target_col)
plt.legend()
plt.tight_layout()
plt.show()

metrics_df["BestOrder"] = str(best_order)
metrics_df.to_csv("paper_arima_metrics.csv", index=False)
pred_frame.to_csv("paper_arima_predictions.csv", index=False)
print("Saved: paper_arima_metrics.csv, paper_arima_predictions.csv")